Imports

In [103]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import io, color, transform
import ipywidgets as widgets
import pydicom
from pydicom.dataset import FileDataset, Dataset
from pydicom.uid import generate_uid
import datetime
import os
import glob

Load image

In [104]:
def loadImage(path, targetSize = (200, 200)):
    if path.lower().endswith('.dcm'):
        ds = pydicom.dcmread(path)
        img = ds.pixel_array.astype(float)
    else:
        img = io.imread(path)
        if len(img.shape) == 3:
            img = color.rgb2gray(img)
        img = img.astype(float)

    min_val = np.min(img)
    max_val = np.max(img)
    if max_val > min_val:
        img = (img - min_val) / (max_val - min_val)
    else:
        img = np.zeros_like(img)
        
    img = transform.resize(img, targetSize)
    return img

Save as DICOM

In [105]:
def saveAsDcm(image, file, patientName, patientID, studyDate, comments):
    fileMeta = Dataset()
    fileMeta.MediaStorageSOPClassUID = pydicom.uid.CTImageStorage
    fileMeta.MediaStorageSOPInstanceUID = generate_uid()
    fileMeta.TransferSyntaxUID = pydicom.uid.ExplicitVRLittleEndian

    ds = FileDataset(file, {}, file_meta=fileMeta, preamble=b"\0" * 128)

    ds.is_little_endian = True
    ds.is_implicit_VR = False

    ds.SOPClassUID = pydicom.uid.CTImageStorage
    ds.SOPInstanceUID = fileMeta.MediaStorageSOPInstanceUID

    ds.PatientName = patientName
    ds.PatientID = patientID
    ds.ImageComments = comments
    ds.StudyDate = studyDate.strftime('%Y%m%d')

    ds.Modality = "CT"
    ds.SeriesInstanceUID = pydicom.uid.generate_uid()
    ds.StudyInstanceUID = pydicom.uid.generate_uid()
    ds.FrameOfReferenceUID = pydicom.uid.generate_uid()

    ds.BitsStored = 8
    ds.BitsAllocated = 8
    ds.SamplesPerPixel = 1
    ds.HighBit = 7

    ds.ImagesInAcquisition = 1
    ds.InstanceNumber = 1

    ds.Rows = image.shape[0]
    ds.Columns = image.shape[1]

    ds.ImageType = r"ORIGINAL\PRIMARY\AXIAL"

    ds.PhotometricInterpretation = "MONOCHROME2"
    ds.PixelRepresentation = 0

    imageUint8 = (image * 255).astype(np.uint8)

    pydicom.dataset.validate_file_meta(ds.file_meta, enforce_standard=True)
    ds.PixelData = imageUint8.tobytes()

    ds.save_as(file, write_like_original=False)


Bresenham's Algorithm

In [106]:
def bresenham(x0, y0, x1, y1):
    points = []
    dX = abs(x1 - x0)
    dY = abs(y1 - y0)

    if x0 < x1:
        sx = 1
    else:
        sx = -1
    
    if y0 < y1:
        sy = 1
    else:
        sy = -1

    err = dX - dY

    while True:
        points.append((x0, y0))
        if x0 == x1 and y0 == y1:
            break

        err2 = err * 2

        if err2 >= -dY:
            err -= dY
            x0 += sx

        if err2 <= dX:
            err += dX
            y0 += sy

    return points

Emiters and detectors positions

In [107]:
def getPositions(alpha, n, l, r, centerX, centerY):
    rad = np.radians(alpha)
    emitters = []
    detectors = []

    offsets = np.linspace(-l/2, l/2, n)

    for s in offsets:
        eX = centerX + r * np.cos(rad) - s * np.sin(rad)
        eY = centerY + r * np.sin(rad) + s * np.cos(rad)

        dX = centerX - r * np.cos(rad) - s * np.sin(rad)
        dY = centerY - r * np.sin(rad) + s * np.cos(rad)

        emitters.append((int(eX), int(eY)))
        detectors.append((int(dX), int(dY)))

    return emitters, detectors

Radon Transform

In [108]:
def radonTransform(image, deltaAlpha, n, l, progress = 100.0):
    height, width = image.shape
    centerX = width // 2
    centerY = height // 2
    r = np.sqrt(centerX ** 2 + centerY ** 2) + 10

    angles = np.arange(0, 181, deltaAlpha)
    sinogram = np.zeros((len(angles), n))

    idx = int((progress / 100.0) * (len(angles) - 1))

    for i in range(idx + 1):
        alpha = angles[i]
        emitters, detectors = getPositions(alpha, n, l, r, centerX, centerY)

        for j in range(n):
            eX, eY = emitters[j]
            dX, dY = detectors[j]

            path = bresenham(eX, eY, dX, dY)
            raySum = 0

            for pX, pY in path:
                if 0 <= pX < width and 0 <= pY < height:
                    raySum += image[pY, pX]

            sinogram[i, j] = raySum

    if np.max(sinogram) > 0:
        sinogram = sinogram / np.max(sinogram)

    return sinogram, angles, idx

Sinogram Visualization

In [109]:
def sinogramVisualization(image, deltaAlpha, n, l, progress):
    sinogram, angles, idx = radonTransform(image, deltaAlpha, n, l, progress)
    currAngle = angles[idx]

    height, width = image.shape
    centerX, centerY = width // 2, height // 2
    r = np.sqrt(centerX ** 2 + centerY ** 2) + 10

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))
    ax1.imshow(image, cmap='gray')
    ax1.set_title("Obraz (Kąt: " + str(currAngle) + "°)")
    
    emitters, detectors = getPositions(currAngle, n, l, r, centerX, centerY)
    stepDraw = max(1, n // 15)
    for j in range(0, n, stepDraw):
        ax1.plot([emitters[j][0], detectors[j][0]], [emitters[j][1], detectors[j][1]], color='red', alpha=0.3, linewidth=0.5)
    
    ax2.imshow(sinogram.T, cmap='gray', aspect='auto', extent=[0, 180, n, 0])
    ax2.set_title("Sinogram")
    ax2.set_xlabel("Nr detektora")
    ax2.set_ylabel("Kąt (stopnie)")
    
    plt.tight_layout()
    plt.show()

Choose Format

In [110]:
style = {'description_width': 'initial'}

formatDropdown = widgets.Dropdown(options=['JPG', 'DICOM'], value='JPG', description='Format')
display(widgets.HTML("<h3>Wybierz format pliku:</h3>"))
display(formatDropdown)

HTML(value='<h3>Wybierz format pliku:</h3>')

Dropdown(description='Format', options=('JPG', 'DICOM'), value='JPG')

Choose Image

In [122]:
chosenFormat = formatDropdown.value

if chosenFormat == 'JPG':
    availableFiles = glob.glob('obrazy/*.jpg')
else:
    availableFiles = glob.glob('obrazy/*.dcm')

filesOptions = [(os.path.basename(f), f) for f in availableFiles]

fileDropdown = widgets.Dropdown(options=filesOptions, description='Wybierz plik:', style=style)
display(widgets.HTML("<h3>Wybierz plik:</h3>"))
display(fileDropdown)

HTML(value='<h3>Wybierz plik:</h3>')

Dropdown(description='Wybierz plik:', options=(('CT_ScoutView-large.dcm', 'obrazy\\CT_ScoutView-large.dcm'), (…

Interactive Sinogram

In [ ]:
image = loadImage(fileDropdown.value)

height, width = image.shape
diagonal = int(np.sqrt(height**2 + width**2))

deltaAlphaSlider = widgets.FloatSlider(value=1.0, min=0.5, max=10.0, step=0.5, description='Krok Δα (deg):', style=style)
nSlider = widgets.IntSlider(value=400, min=10, max=diagonal, step=10, description='Liczba detektorów (n):', style=style)
lSlider = widgets.FloatSlider(value=diagonal, min=50.0, max=diagonal*1.5, step=10.0, description='Rozpiętość (l):', style=style)
progressSlider = widgets.IntSlider(value=100, min=0, max=100, step=1, description='Postęp obrotu (%):', style=style)

widgets.interact(sinogramVisualization, image=widgets.fixed(image), deltaAlpha=deltaAlphaSlider, n=nSlider, l=lSlider, progress=progressSlider)

if chosenFormat == 'DICOM':
    try:
        ds = pydicom.dcmread(fileDropdown.value)
        print("Metadane DICOM")
        print("Pacjent: " + str(ds.get('PatientName', 'Brak danych')))
        print("ID Pacjenta: " + str(ds.get('PatientID', 'Brak danych')))
        print("Data badania: " + str(ds.get('StudyDate', 'Brak danych')))
        print("Komentarz: " + str(ds.get('ImageComments', ds.get('StudyComments', 'Brak danych'))))
    except Exception as dicom_e:
        print(f"Nie udało się odczytać nagłówka DICOM: {dicom_e}")

interactive(children=(FloatSlider(value=1.0, description='Krok Δα (deg):', max=10.0, min=0.5, step=0.5, style=…

Metadane DICOM
Pacjent: Doe^John
ID Pacjenta: 42
Data badania: Brak danych
Komentarz: what a lovely image we have here...


Save sinogram as variable

In [124]:
finalDeltaAlpha = deltaAlphaSlider.value
finalN = nSlider.value
finalL = lSlider.value

sinogram, _, _ = radonTransform(image, finalDeltaAlpha, finalN, finalL)

Ramp Filter

In [114]:
def generateFilter(size):
    if size % 2 == 0:
        size += 1

    center = size // 2
    mask = np.zeros(size)

    for i in range(size):
        k = i - center
        if k == 0:
            mask[i] = 1.0
        elif k % 2 != 0:
            mask[i] = -4.0 / (np.pi ** 2 * k ** 2)

    return mask

Apply Filter to Sinogram

In [115]:
def applyFilter(sinogram, maskSize):
    mask = generateFilter(maskSize)
    filtered = np.zeros_like(sinogram)

    for i in range(sinogram.shape[0]):
        filtered[i, :] = np.convolve(sinogram[i, :], mask, mode='same')

    return filtered

Back Projection

In [127]:
def backProjection(sinogram, deltaAlpha, n, l, targetShape, progress=100.0, useFilter=False, maskSize=21):
    height, width = targetShape
    centerX, centerY = width // 2, height // 2
    r = np.sqrt(centerX ** 2 + centerY ** 2) + 10

    angles = np.arange(0, 180, deltaAlpha)

    reconstructed = np.zeros((height, width))
    counts = np.zeros((height, width))

    if useFilter == True:
        displaySinogram = applyFilter(sinogram, maskSize)
    else:
        displaySinogram = sinogram

    idx = int((progress / 100.0) * (len(angles) - 1))
    for i in range(idx + 1):
        alpha = angles[i]
        emitters, detectors = getPositions(alpha, n, l, r, centerX, centerY)

        for j in range(n):
            rayVal = displaySinogram[i, j]
            eX, eY = emitters[j]
            dX, dY = detectors[j]

            path = bresenham(eX, eY, dX, dY)

            for pX, pY in path:
                if 0 <= pX < width and 0 <= pY < height:
                    reconstructed[pY, pX] += rayVal
                    counts[pY, pX] += 1

    np.divide(reconstructed, counts, out=reconstructed, where=counts!=0)

    reconstructed[reconstructed < 0] = 0
    if np.max(reconstructed) > 0:
        reconstructed = reconstructed / np.max(reconstructed)

    return reconstructed, angles, idx

Reconstruction Visualization

In [128]:
def reconstructionVisualization(sinogram, deltaAlpha, n, l, targetShape, progress, useFilter, maskSize):
    if useFilter == True:
        displaySinogram = applyFilter(sinogram, maskSize)
    else:
        displaySinogram = sinogram

    reconstructed, angles, idx = backProjection(sinogram, deltaAlpha, n, l, targetShape, progress, useFilter, maskSize)

    currAngle = angles[idx]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))

    ax1.imshow(displaySinogram.T, cmap='gray', aspect='auto', extent=[0, 180, n, 0])
    ax1.axvline(x=currAngle, color='red', linewidth=2)
    ax1.set_title("Sinogram")
    ax1.set_ylabel("Kąt (stopnie)")
    ax1.set_xlabel("Nr detektora")

    ax2.imshow(reconstructed, cmap='gray')
    ax2.set_title("Obraz wynikowy (Back Projection)")
    
    plt.tight_layout()
    plt.show()

Interactive Reconstruction

In [129]:
progressReconSlider = widgets.IntSlider(value=100, min=0, max=100, step=1, description='Postęp (%):', style=style)
filterCheckbox = widgets.Checkbox(value=False, description="Filtr splotowy: ", style=style)
maskSizeSlider = widgets.IntSlider(value=21, min=5, max=101, step=2, description='Rozmiar maski: ', style=style)

widgets.interact(reconstructionVisualization, sinogram=widgets.fixed(sinogram), deltaAlpha=widgets.fixed(finalDeltaAlpha), n=widgets.fixed(finalN), l=widgets.fixed(finalL), targetShape=widgets.fixed(image.shape), progress=progressReconSlider, useFilter=filterCheckbox, maskSize=maskSizeSlider)

interactive(children=(IntSlider(value=100, description='Postęp (%):', style=SliderStyle(description_width='ini…

<function __main__.reconstructionVisualization(sinogram, deltaAlpha, n, l, targetShape, progress, useFilter, maskSize)>

Save reconstructed image as DICOM

In [ ]:
reconstructed, _, _ = backProjection(sinogram, finalDeltaAlpha, finalN, finalL, image.shape, 100, filterCheckbox.value, maskSizeSlider.value)

filenameForm = widgets.Text(value='wynik_badania.dcm', description='Nazwa pliku:')
patientNameForm = widgets.Text(value="Jan Kowalski", description='Pacjent:')
patientIDForm = widgets.IntText(value=1234, description='ID Pacjenta:')
studyDateForm = widgets.DatePicker(value=datetime.date.today(), description='Data badania:')
commentsForm = widgets.Textarea(value='Badanie rutynowe', description='Komentarz:')

saveButton = widgets.Button(description="Zapisz do DICOM", button_style='success', icon='save')
outputMsg = widgets.Output()

def saveButtonOnClick(b):
    with outputMsg:
        outputMsg.clear_output()
        try:
            saveAsDcm(image=reconstructed, file=filenameForm.value, patientName=patientNameForm.value, patientID=str(patientIDForm.value), studyDate=studyDateForm.value, comments=commentsForm.value)
            print("Plik zapisanano jako " + filenameForm.value)
        except Exception as e:
            print(f"Wystąpił błąd podczas zapisu: {e}")

saveButton.on_click(saveButtonOnClick)

form = widgets.VBox([widgets.HTML("<h3>Zapisz jako DICOM</h3>"), filenameForm, patientNameForm, patientIDForm, studyDateForm, commentsForm, saveButton, outputMsg])
display(form)